# 03 - Load station list

The ingestion job takes its list of stations from the **Oracle data hub** (`stations` table), the single source of truth shared with `dim_station` in dbt.

To keep the hourly ingestion running when Oracle is unavailable, every valid list read from Oracle is cached in `data/state/stations_cache.json`. The departures API keeps no history, so a missed run is lost permanently, while the station list changes rarely: a cached list of a few hours is an acceptable fallback.

## Behaviour

| Oracle | Cache | Result |
|---|---|---|
| returns ≥ 1 station | any | Oracle list used, cache **refreshed** |
| unreachable / query error | present | cached list used, **warning** logged with cache age |
| returns 0 stations | present | treated as a failure: cached list used, cache **not** overwritten |
| unreachable or empty | missing | **error**: the run cannot start |

Only a validated list may overwrite the cache, so an accidental `DELETE` in Oracle cannot destroy the fallback.

In [ ]:
import json
import logging
import os
from datetime import datetime, timezone
from pathlib import Path

import oracledb

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("ingestion")

CACHE_PATH = Path("/workspace/data/state/stations_cache.json")

STATION_QUERY = """
    SELECT station_id, station_name
    FROM stations
    ORDER BY station_id
"""


def _read_oracle(query, dsn, user, password):
    """Return the station list from Oracle as a list of dicts."""
    with oracledb.connect(user=user, password=password, dsn=dsn, tcp_connect_timeout=10) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            return [{"station_id": str(sid), "station_name": name} for sid, name in cur.fetchall()]


def _write_cache(stations, cache_path):
    """Write the cache atomically: a crash during the write never leaves a half-written file."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "cached_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": "oracle:stations",
        "stations": stations,
    }
    tmp_path = cache_path.with_suffix(".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp_path, cache_path)


def _read_cache(cache_path):
    """Return (stations, cached_at) from the cache file, or None if there is no usable cache."""
    if not cache_path.exists():
        return None
    payload = json.loads(cache_path.read_text(encoding="utf-8"))
    if not payload.get("stations"):
        return None
    return payload["stations"], payload["cached_at"]


def load_stations(
    cache_path=CACHE_PATH,
    query=STATION_QUERY,
    dsn=None,
    user=None,
    password=None,
):
    """Load the stations to ingest.

    Returns:
        (stations, source) where source is "oracle" or "cache".
    Raises:
        RuntimeError if Oracle fails (or returns no station) and no cache is available.
    """
    dsn = dsn or os.environ["ORACLE_DSN"]
    user = user or os.environ["ORACLE_APP_USER"]
    password = password or os.environ["ORACLE_APP_PASSWORD"]

    try:
        stations = _read_oracle(query, dsn, user, password)
        if not stations:
            raise ValueError("Oracle returned 0 stations")
        _write_cache(stations, cache_path)
        log.info("Loaded %d stations from Oracle, cache refreshed", len(stations))
        return stations, "oracle"

    except (oracledb.Error, ValueError) as exc:
        oracle_error = f"{type(exc).__name__}: {exc}".splitlines()[0]

    cached = _read_cache(cache_path)
    if cached is None:
        raise RuntimeError(f"Station list unavailable: Oracle failed ({oracle_error}) and no cache at {cache_path}")

    stations, cached_at = cached
    log.warning(
        "Oracle unavailable (%s). Using %d cached stations from %s",
        oracle_error, len(stations), cached_at,
    )
    return stations, "cache"

## Tests
Each test uses a dedicated cache file (`stations_cache_test.json`), deleted at the start.

In [ ]:
# Tests use their own cache file so the production cache is never modified.
TEST_CACHE = Path("/workspace/data/state/stations_cache_test.json")
TEST_CACHE.unlink(missing_ok=True)
UNREACHABLE_DSN = "oracle:1599/FREEPDB1"   # wrong port: simulates Oracle being down

### Test 1 - Oracle reachable
Expected: 16 stations from Oracle, cache file created.

In [ ]:
stations, source = load_stations(cache_path=TEST_CACHE)
print(source, len(stations), stations[0])
print(TEST_CACHE.read_text(encoding="utf-8")[:300])

### Test 2 - Oracle unreachable, cache present
Expected: warning with the cache date, 16 stations from the cache.

In [ ]:
stations, source = load_stations(cache_path=TEST_CACHE, dsn=UNREACHABLE_DSN)
print(source, len(stations))

### Test 3 - Oracle returns 0 stations
Expected: warning, cached list used, `cached_at` unchanged (the empty result did not overwrite the cache).

In [ ]:
cached_at_before = json.loads(TEST_CACHE.read_text(encoding="utf-8"))["cached_at"]
stations, source = load_stations(cache_path=TEST_CACHE, query=STATION_QUERY.replace("ORDER BY", "WHERE 1 = 0 ORDER BY"))
cached_at_after = json.loads(TEST_CACHE.read_text(encoding="utf-8"))["cached_at"]
print(source, len(stations), "| cache unchanged:", cached_at_before == cached_at_after)

### Test 4 - Oracle unreachable, no cache
Expected: `RuntimeError: Station list unavailable`.

In [ ]:
TEST_CACHE.unlink(missing_ok=True)
load_stations(cache_path=TEST_CACHE, dsn=UNREACHABLE_DSN)